# DDR ground-truth lesion probe — frozen P, E1 and E2 encoders against expert lesion masks (record §68)

**Question.** Does E1's frozen representation contain more information predictive of real, expert-annotated lesions than P's?

**No grader is trained and no encoder is changed.** For seeds 42 / 123 / 2026, the pinned BEST encoders of P, E1 and E2 are frozen;
the same fresh 1×1 probe is trained on the DDR training split (383 images) to predict, per 16×16 cell, whether an expert-annotated
MA / HE / EX / SE lesion is present.

- **Five-epoch probe** — the protocol of §63.2 unchanged (Adam 1e-3, batch 16, 5 epochs). This is the pre-declared primary result.
- **Converged probe** — the same probe continued until the DDR **validation** loss stops improving (patience 5, min-delta 1e-4,
  cap 100 epochs); reported next to the five-epoch result as a robustness analysis, never instead of it.
- **Test split (225 images)** — predicted exactly twice per probe (once per weight set) after the stopping decision is final.
  Nothing is chosen from test performance.

Probe set: 756 images (383 / 148 / 225); `007-5869-300` is excluded (§66.1). Stage 2 is applied once, then the locked 512 frame.
GPU runtime (T4). The run is resumable: finished (model, seed) pairs are kept. The paired bootstrap and the criterion
(E1 − P positive in 3/3 seeds and the 95 % interval excluding zero) are computed afterwards from the saved test logits.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import ddr_probe as dp, e1_probe as ep
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: the locked probe set and the nine pinned checkpoints (nothing is trained here) ====
EXP = colab_config.DRIVE.experiments_root
DRIVE_ROOT = colab_config.DRIVE.project_root
DDR_LESION = posixpath.join(colab_config.DATASET_ROOT, "DDR", "raw", "lesion_segmentation")
DDR_AUDIT = posixpath.join(colab_config.DATASET_ROOT, "DDR", "audit")
OUT_DIR = posixpath.join(EXP, "DDRProbe", "ddr_ground_truth_probe_v1")
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
MANIFEST = dp.read_manifest(DDR_AUDIT, DDR_LESION)       # pinned manifest; SHA-256 of all 756 images; every mask present
CHECKPOINTS, PRETRAINED, _ = dp.pinned_checkpoints(REPO_DIR, DRIVE_ROOT)
print("probe set:", {s: len(v) for s, v in MANIFEST.items()}, "| excluded:", dp.EXCLUDED, "| manifest", dp.MANIFEST_SHA256[:16])
for (_m, _s), (_p, _sha) in sorted(CHECKPOINTS.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    print(f"  {_m:<2} seed {_s:<5} {_sha}")
print("probe:", ep.PROBE)
print("stopping (validation only):", dp.STOPPING)
print("contrasts:", dp.CONTRASTS, "| primary:", dp.PRIMARY, "| bootstrap:", dp.N_BOOT, "resamples, seed", dp.BOOT_SEED)

In [ ]:
# ==== [3] THE PROBE RUN -- frozen encoders, probes only. Resumable. ====
RUN_DDR_PROBE = True
if RUN_DDR_PROBE:
    SUMMARY = dp.run(DRIVE_ROOT, DDR_LESION, DDR_AUDIT, OUT_DIR, repo_dir=REPO_DIR)
    print("\nDDR training-target prior (MA, HE, EX, SE):", [round(p, 4) for p in SUMMARY["prior"]])
    print("positive-cell rate per split:", {s: [round(v, 4) for v in r] for s, r in SUMMARY["positive_cell_rate"].items()})
    for _key in sorted(SUMMARY["results"], key=lambda k: (int(k.split("seed")[1]), k)):
        _r = SUMMARY["results"][_key]
        _b = _r["behaviour"]
        print(f"\n===== {_key} | converged at epoch {_b['converged_epoch']} of {_b['epochs_run']} ({_b['stopped_by']})")
        for _v in dp.VARIANTS:
            print(f"  {_v:<10} test", {c: round(x, 4) for c, x in _r[_v]["test"]["scores"].items()},
                  "| val", round(_r[_v]["val"]["scores"]["mean"], 4), "| train", round(_r[_v]["train"]["scores"]["mean"], 4))
        print("  val loss per epoch:", [(c["epoch"], round(c["val_loss"], 4)) for c in _b["curve"] if c["epoch"] <= 5 or c["epoch"] % 10 == 0 or c["epoch"] == _b["epochs_run"]])
    print("\nsaved ->", OUT_DIR)
else:
    print("probe not run (RUN_DDR_PROBE = False)")